# o2org: full walkthrough

This notebook exercises the whole `o2org` package end to end:

1. Connecting (auth via email/password)
2. Retrieving data (`list()` / `get()` — the common case)
3. Other procedures: creating & updating records
4. Handling errors

**About the data in this notebook:** `BASE_URL`/`VALID_EMAIL`/`VALID_PASSWORD`
below point at a real org_dma instance (a local `php artisan serve` at
`http://localhost:8000`), not a mock. It now implements the endpoints in the
README's *API contract* section (Sanctum login, full create/update routes),
so re-running this notebook top to bottom talks to a real Postgres-backed
Laravel app. The Retrieving-data section reads whatever compounds/projects
already exist in that database, so exact ids/counts will differ from a fresh
install — the code (not the specific values) is what to focus on.

## Environment setup (do this once)

This notebook needs `o2org` installed in whatever Python environment your
Jupyter kernel is running — it is **not** enough to just open this file.

From the repo root:

```bash
python3 -m venv .venv
source .venv/bin/activate        # Windows: .venv\Scripts\activate
pip install -e ".[notebook]"     # installs o2org plus jupyter/ipykernel
```

Then point Jupyter at that same environment:

- **VS Code**: click *Select Kernel* (top right of this notebook) → choose the
  `.venv` interpreter for this repo (create it first if it isn't listed —
  VS Code will offer to register it after you activate it once in a terminal).
- **Jupyter Notebook/Lab**: `Kernel` menu → `Change Kernel` → the one created
  by the steps above.

If the `import o2org` cell below fails with `ModuleNotFoundError`, the kernel
is not pointed at that environment yet — fix that before continuing, the rest
of the notebook assumes it.

## Setup

In [1]:
import pandas as pd

import o2org
from o2org import AuthenticationError, NotFoundError, ValidationError

## 0. Server connection settings

Point these at your org_dma instance.

In [2]:
BASE_URL = "http://localhost:8000"
VALID_EMAIL = "anna@example.com"
VALID_PASSWORD = "admin"

## 1. Connecting

In [3]:
client = o2org.connect(
    base_url=BASE_URL,
    email=VALID_EMAIL,
    password=VALID_PASSWORD,
)
print(client)
print("authenticated:", client.is_authenticated)

<OrgClient http://localhost:8000 as anna@example.com>
authenticated: True


Credentials can also come from environment variables (`O2ORG_BASE_URL`,
`O2ORG_EMAIL`, `O2ORG_PASSWORD`) or `~/.o2org/config.ini`, so `o2org.connect()`
alone works with no arguments at all — see the README for the config file
format. Use `connect()` as a context manager to log out automatically:

In [4]:
with o2org.connect(base_url=BASE_URL, email=VALID_EMAIL, password=VALID_PASSWORD) as scoped_client:
    print("inside the with-block:", scoped_client.is_authenticated)
print("after the with-block:", scoped_client.is_authenticated)

inside the with-block: True
after the with-block: False


## 2. Retrieving data

This is what you'll use most. Every table is an attribute on `client`. `.list()`
is lazy — it never touches the network by itself. `.count()`/`.pages` make one
cheap request and read the server's own pagination total, no matter how many
pages that total spans; `.get()` (or iterating/indexing it directly) is what
actually fetches the records, as plain Python data — no DataFrame conversion
step to remember.

In [5]:
first_page = client.compounds.list()
print(f"{len(first_page)} compounds on the first page:")
for row in first_page:
    print(" -", row["canonical_name"])

20 compounds on the first page:
 - Fumaric acid, butyl 3,4-dimethoxyphenyl ester
 - Eicosadienoic acid
 - 3-Methoxytyramine
 - (S)-3-Hydroxyisobutyric acid
 - Ureidopropionic acid
 - Androsterone
 - Carnosine
 - Dihydrobiopterin
 - Silane, dimethyl(tetrahydrofurfuryloxy)propoxy-
 - 2-Hydroxybutyric acid, TBDMS
 - 3-Hydroxybutyric acid, mono-TBDMS # 1
 - 2-Butenoic acid, 2-[(trimethylsilyl)oxy]-, trimethylsilyl ester
 - Acetoacetic acid, bis(trimethylsilyl)- deriv.
 - Ethanol, 2-[2-(2-butoxyethoxy)ethoxy]-
 - 3-O-Sulfogalactosylceramide (d18:1/24:0)
 - Threono-1,4-lactone, di-TMS
 - Threono-1,4-lactone, TMS
 - Erythronic acid, 1,4-lactone, TMS
 - Pentanamide, N-(4-methoxyphenyl)-
 - 3-Methylbut-3-en-1-yl cinnamate


In [6]:
client.compounds.list(like="canonical_name,acid").count()

30645

In [7]:
client.compounds.get(60)

{'id': 60,
 'canonical_name': '2-Ketobutyric acid',
 'iupac_name': '2-oxobutanoic acid',
 'molecular_formula': 'C4H6O3',
 'smiles': 'CCC(=O)C(O)=O',
 'inchi': 'InChI=1S/C4H6O3/c1-2-3(5)4(6)7/h2H2,1H3,(H,6,7)',
 'inchikey': 'TYEYBOSBBBHJIV-UHFFFAOYSA-N',
 'pubchem_cid': '58',
 'cas': '600-18-0',
 'hmdb_id': 'HMDB0000005',
 'chebi_id': 'CHEBI:30831',
 'synonyms': [{'id': 2138,
   'name': '2-Ketobutanoic acid',
   'source': {'id': 1,
    'name': 'HMDB',
    'created_at': '2026-05-04T03:24:38.000000Z',
    'updated_at': '2026-05-04T03:24:38.000000Z'},
   'source_id': 1,
   'created_at': '2026-05-04T03:30:47.000000Z',
   'updated_at': '2026-05-04T03:30:47.000000Z'},
  {'id': 2139,
   'name': '2-Oxobutyric acid',
   'source': {'id': 1,
    'name': 'HMDB',
    'created_at': '2026-05-04T03:24:38.000000Z',
    'updated_at': '2026-05-04T03:24:38.000000Z'},
   'source_id': 1,
   'created_at': '2026-05-04T03:30:47.000000Z',
   'updated_at': '2026-05-04T03:30:47.000000Z'},
  {'id': 2140,
   'name':

In [8]:
print(client.projects.list().names())
print(client.projects.list().count())
client.projects.list()

['test', 'PD Volatile Compounds GC-MS', 'Essential Oils', 'Milk Traceability Study']
4


[{'id': 2, 'name': 'test', 'description': None, 'status': 'active', 'user_id': 1, 'project_compounds': [{'id': 5, 'project_id': 2, 'compound_id': 77743, 'custom_name': 'Fruktoza', 'is_duplicate': False, 'mz': '180.1600000000', 'rt': '3000.0000000000', 'is_mapped': True, 'notes': None, 'compound': {'id': 77743, 'canonical_name': 'D-Fructose', 'iupac_name': '(3S,4R,5R)-1,3,4,5,6-pentahydroxyhexan-2-one', 'molecular_formula': 'C6H12O6', 'smiles': 'C([C@H]([C@H]([C@@H](C(=O)CO)O)O)O)O', 'inchi': 'InChI=1S/C6H12O6/c7-1-3(9)5(11)6(12)4(10)2-8/h3,5-9,11-12H,1-2H2/t3-,5-,6-/m1/s1', 'inchikey': 'BJHIKXHVCXFQLS-UYFOZJQFSA-N', 'pubchem_cid': '5984', 'cas': '30237-26-4', 'hmdb_id': 'HMDB0000660', 'chebi_id': 'CHEBI:15824', 'synonyms': [{'id': 15028, 'name': 'Bieffe brand OF fructose', 'source': {'id': 1, 'name': 'HMDB', 'created_at': '2026-05-04T03:24:38.000000Z', 'updated_at': '2026-05-04T03:24:38.000000Z'}, 'source_id': 1, 'created_at': '2026-05-04T15:48:52.000000Z', 'updated_at': '2026-05-04T15

In [9]:
project_compounds = client.project_compounds.list(project_id=2)
print(f"{len(project_compounds)} compounds for project 2")
project_compounds[:3]

8 compounds for project 2


[{'id': 5,
  'project_id': 2,
  'compound_id': 77743,
  'custom_name': 'Fruktoza',
  'is_duplicate': False,
  'mz': '180.1600000000',
  'rt': '3000.0000000000',
  'is_mapped': True,
  'notes': None,
  'compound': {'id': 77743,
   'canonical_name': 'D-Fructose',
   'iupac_name': '(3S,4R,5R)-1,3,4,5,6-pentahydroxyhexan-2-one',
   'molecular_formula': 'C6H12O6',
   'smiles': 'C([C@H]([C@H]([C@@H](C(=O)CO)O)O)O)O',
   'inchi': 'InChI=1S/C6H12O6/c7-1-3(9)5(11)6(12)4(10)2-8/h3,5-9,11-12H,1-2H2/t3-,5-,6-/m1/s1',
   'inchikey': 'BJHIKXHVCXFQLS-UYFOZJQFSA-N',
   'pubchem_cid': '5984',
   'cas': '30237-26-4',
   'hmdb_id': 'HMDB0000660',
   'chebi_id': 'CHEBI:15824',
   'synonyms': [{'id': 15028,
     'name': 'Bieffe brand OF fructose',
     'source': {'id': 1,
      'name': 'HMDB',
      'created_at': '2026-05-04T03:24:38.000000Z',
      'updated_at': '2026-05-04T03:24:38.000000Z'},
     'source_id': 1,
     'created_at': '2026-05-04T15:48:52.000000Z',
     'updated_at': '2026-05-04T15:48:52.00

## 3. Pipelines

`o2org.pipelines` (install with `pip install -e ".[pipelines]"`) adds
domain-specific statistical/ML pipelines on top of everything above — one
pair of `stats_pipeline()`/`ml_pipeline()` functions per result family
(stable isotopes, elemental composition, MK GC-MS/IRMS, VOC GC-MS/IRMS).

Every pipeline starts by extracting one `record_type` for one project out of
`experiment_records.details` (EAV JSON) and pivoting it long -> wide (one row
per sample, one column per analyte).

In [10]:
from o2org.pipelines.utils import extract, pivot_wide

samples = client.samples.list(project_id=12)
print(f"{len(samples)} samples in project 12:")
for s in samples:
    print(" -", s["id"], s["sample_group"], s["sample_subgroup"])

13 samples in project 12:
 - 6 plant None
 - 12 plant None
 - 24 food fats_oils
 - 25 plant None
 - 26 plant None
 - 27 plant None
 - 28 plant None
 - 29 environment water
 - 30 environment soil
 - 31 environment air
 - 32 environment sediments
 - 33 environment biological
 - 34 environment surface


In [11]:
df_long = extract(client, "result_stable_isotopes", project_id=12)
df_long[["sample_id", "sample_group", "details_analyte", "details_value", "details_unit"]]

,sample_id,sample_group,details_analyte,details_value,details_unit
0,6,plant,d13c,-24.80,‰
1,6,plant,d18o,26.10,‰
2,6,plant,d15n,8.09,‰
3,6,plant,d34s,4.69,‰
4,12,plant,d13c,-24.77,‰
5,12,plant,d15n,8.28,‰
6,12,plant,d18o,25.02,‰
7,12,plant,d34s,4.61,‰
8,25,plant,d13c,-24.96,‰
9,25,plant,d15n,7.91,‰


In [12]:
df_wide = pivot_wide(df_long, index_cols=["sample_id", "sample_group"], analyte_col="details_analyte")
df_wide

,sample_id,sample_group,d13c,d15n,d18o,d34s
0,6,plant,-24.80,8.09,26.10,4.69
1,12,plant,-24.77,8.28,25.02,4.61
2,25,plant,-24.96,7.91,25.99,4.74
3,26,plant,-24.74,8.23,26.03,5.34
4,27,plant,-24.86,7.74,26.18,4.71
5,28,plant,-24.74,7.99,25.91,4.80
6,29,environment,-19.63,11.95,19.79,8.89
7,30,environment,-19.84,12.11,20.21,9.13
8,31,environment,-19.36,11.88,19.74,8.76
9,32,environment,-19.82,12.34,19.94,8.75


The isotopes pipeline runs this extraction/pivot step itself, then chains
**Shapiro-Wilk normality → Kruskal-Wallis/ANOVA → Bonferroni/FDR correction →
box/KDE plots → isotope biplot → PCA**. One call already returns the full
report — p-values, verdicts, and every figure — as the last expression in a
cell:

In [13]:
from o2org.pipelines import isotopes

isotopes.stats_pipeline(client, project_id=12)

feature,test,statistic,pvalue,pvalue_corrected,significant
d15n,anova,1428.789952,4.002903e-12,1.601161e-11,True
d13c,anova,1213.745789,8.997638e-12,1.799528e-11,True
d34s,anova,990.927136,2.459824e-11,3.279765e-11,True
d18o,kruskal,8.307692,3.947752e-03,3.947752e-03,True
sample_group,n,shapiro_stat,shapiro_pval,is_normal,analyte
environment,6,0.941921,0.674713,True,d13c
plant,6,0.866372,0.212121,True,d13c
environment,6,0.981511,0.958815,True,d15n
plant,6,0.966873,0.870809,True,d15n
environment,6,0.955702,0.786082,True,d18o


It's still a plain dict underneath, so picking out one piece works exactly
like before (e.g. for further analysis, or to save a specific figure):

In [14]:
result = isotopes.stats_pipeline(client, project_id=12)
result["tests"]

,feature,test,statistic,pvalue,pvalue_corrected,significant
0,d15n,anova,1428.789952,4.002903e-12,1.601161e-11,True
1,d13c,anova,1213.745789,8.997638e-12,1.799528e-11,True
2,d34s,anova,990.927136,2.459824e-11,3.279765e-11,True
3,d18o,kruskal,8.307692,3.947752e-03,3.947752e-03,True


In [15]:
result["figures"]["isotope_biplot"].show()

Set `include_figures=False` to skip building/embedding plots entirely — just the data and test results, faster for headless/batch use:

In [16]:
isotopes.stats_pipeline(client, project_id=12, include_figures=False)

feature,test,statistic,pvalue,pvalue_corrected,significant
d15n,anova,1428.789952,4.002903e-12,1.601161e-11,True
d13c,anova,1213.745789,8.997638e-12,1.799528e-11,True
d34s,anova,990.927136,2.459824e-11,3.279765e-11,True
d18o,kruskal,8.307692,3.947752e-03,3.947752e-03,True
sample_group,n,shapiro_stat,shapiro_pval,is_normal,analyte
environment,6,0.941921,0.674713,True,d13c
plant,6,0.866372,0.212121,True,d13c
environment,6,0.981511,0.958815,True,d15n
plant,6,0.966873,0.870809,True,d15n
environment,6,0.955702,0.786082,True,d18o


## 4. Statistics & ML toolkit, applied manually

Every pipeline is built from standalone functions in `o2org.pipelines.stats`
/ `ml` / `preprocessing` / `viz` — worth knowing since your own analysis
won't always fit one of the four pre-built pipelines. Project 12 only has a
single sample, too few to show a real group comparison, so this section
builds a small synthetic isotope-shaped dataset (same column shape a real
`extract()`/`pivot_wide()` call would give you) purely to demonstrate each
function with real, readable output.

In [17]:
import numpy as np

from o2org.pipelines import ml, preprocessing, stats, viz

rng = np.random.default_rng(0)
demo_df = pd.DataFrame({
    "sample_id": range(1, 41),
    "sample_group": ["food"] * 20 + ["environment"] * 20,
    "instrument": ["gc-a", "gc-b"] * 20,
    "d13c": np.concatenate([rng.normal(-25, 0.5, 20), rng.normal(-20, 0.5, 20)]),
    "d15n": np.concatenate([rng.normal(8, 0.5, 20), rng.normal(12, 0.5, 20)]),
})
demo_df.head()

,sample_id,sample_group,instrument,d13c,d15n
0,1,food,gc-a,-24.937135,7.370467
1,2,food,gc-b,-25.066052,8.756962
2,3,food,gc-a,-24.679789,8.672938
3,4,food,gc-b,-24.947550,8.390656
4,5,food,gc-a,-25.267835,8.132228


### Normality & group comparison

In [ ]:
stats.descriptive_stats(demo_df, "d13c", "sample_group")

,sample_group,mean,median,std,n
0,environment,-19.968855,-20.008631,0.35726,20
1,food,-25.091589,-25.087724,0.43649,20


In [19]:
stats.shapiro_by_group(demo_df, "d13c", "sample_group")

,sample_group,n,shapiro_stat,shapiro_pval,is_normal
0,environment,20,0.962604,0.597074,True
1,food,20,0.974651,0.848348,True


In [20]:
slope, intercept, r = stats.qq_regression(demo_df["d13c"])
print(f"Q-Q regression: slope={slope:.3f} intercept={intercept:.3f} r={r:.4f}")

Q-Q regression: slope=2.401 intercept=-22.530 r=0.8870


In [21]:
stats.kruskal_wallis(demo_df, "d13c", "sample_group")

{'statistic': np.float64(29.26829268292684),
 'pvalue': np.float64(6.301848221392258e-08)}

In [22]:
stats.anova(demo_df, "d13c", "sample_group")

{'statistic': np.float64(1649.6463144185122),
 'pvalue': np.float64(6.477806796942816e-33)}

In [23]:
print("Bonferroni:", stats.bonferroni_correction([0.001, 0.02, 0.03, 0.5]))
print("BH-FDR:    ", stats.benjamini_hochberg([0.001, 0.02, 0.03, 0.5]))

Bonferroni: [0.004 0.08  0.12  1.   ]
BH-FDR:     [0.004 0.04  0.04  0.5  ]


In [ ]:
tests = stats.test_all_features(demo_df, ["d13c", "d15n"], "sample_group", correction="fdr_bh")
tests

,feature,test,statistic,pvalue,pvalue_corrected,significant
0,d13c,anova,1649.646314,6.477807e-33,1.295561e-32,True
1,d15n,anova,506.528579,1.430818e-23,1.430818e-23,True


### Preprocessing & normalization

In [25]:
imputed, imputed_counts = preprocessing.lod_half_impute(demo_df, ["d13c", "d15n"], batch_col="instrument")
normalized = preprocessing.batch_median_normalize(imputed, ["d13c", "d15n"], batch_col="instrument")
print("imputed counts:", imputed_counts.to_dict())
normalized.head()

imputed counts: {'d13c': 0, 'd15n': 0}


,sample_id,sample_group,instrument,d13c,d15n
0,1,food,gc-a,1.111954,0.716621
1,2,food,gc-b,1.115544,0.881054
2,3,food,gc-a,1.100479,0.843259
3,4,food,gc-b,1.110270,0.844199
4,5,food,gc-a,1.126700,0.790686


In [26]:
outlier_mask = preprocessing.grubbs_outliers(demo_df["d13c"].values)
print(f"{outlier_mask.sum()} outliers flagged out of {len(outlier_mask)} values")

0 outliers flagged out of 40 values


### Dimensionality reduction, classification & clustering

In [27]:
pca_result = ml.pca(demo_df, columns=["d13c", "d15n"])
print("explained variance ratio:", pca_result.explained_variance_ratio.round(3))
pca_result.scores.head()

explained variance ratio: [0.976 0.024]


,PC1,PC2
0,-2.392073,-0.463105
1,-1.814087,0.311244
2,-1.674025,0.109614
3,-1.927721,0.072146
4,-2.194747,0.068563


In [28]:
rf = ml.random_forest_classify(demo_df, ["d13c", "d15n"], "sample_group", positive_class="food", negative_class="environment")
accuracy = (rf.y_pred == rf.y_true).mean()
print(f"cross-validated accuracy: {accuracy:.2%}")

importances = ml.mdi_importance(rf.model, ["d13c", "d15n"])
cutoff, elbow_idx = ml.elbow_cutoff(importances["importance"].values)
importances

cross-validated accuracy: 100.00%


,feature,importance
0,d15n,0.51
1,d13c,0.49


In [29]:
cluster_result = ml.kmeans_cluster(demo_df, ["d13c", "d15n"], k=2)
print("cluster sizes:", pd.Series(cluster_result.labels).value_counts().to_dict())

lda_result = ml.lda_classify(demo_df, ["d13c", "d15n"], "sample_group")
lda_accuracy = (lda_result.encoder.inverse_transform(lda_result.y_pred) == demo_df["sample_group"].values).mean()
print(f"LDA accuracy: {lda_accuracy:.2%}")

cluster sizes: {0: 20, 1: 20}
LDA accuracy: 100.00%


### Visualization

Every function below returns a Plotly figure — `.show()` renders it inline.

In [30]:
viz.box_plot(demo_df, ["d13c", "d15n"], "sample_group").show()

In [31]:
viz.pca_score_scatter(pca_result, group=demo_df["sample_group"]).show()

In [32]:
viz.roc_plot(rf.y_true, rf.y_proba).show()

In [33]:
viz.mdi_elbow_plot(importances["importance"].values, elbow_idx, cutoff).show()